In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

basilsaji03_unisign_dataset_path = kagglehub.dataset_download('basilsaji03/unisign-dataset')
basilsaji03_repo_files_path = kagglehub.dataset_download('basilsaji03/repo-files')
basilsaji03_s2s_dataset_phrases_path = kagglehub.dataset_download('basilsaji03/s2s-dataset-phrases')
basilsaji03_zora_phrase_dataset_path = kagglehub.dataset_download('basilsaji03/zora-phrase-dataset')
basilsaji03_phrase_ds_split_files_path = kagglehub.dataset_download('basilsaji03/phrase-ds-split-files')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Cell 1 — Imports

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from tqdm import tqdm

Cell 2 — Paths

In [ ]:
MEETIX_DATASET = "/kaggle/input/datasets/basilsaji03/phrase-ds-split-files/meetix_dataset"

LANDMARK_ROOT = "/kaggle/input/datasets/basilsaji03/zora-phrase-dataset/extracted_landmarks"

REPO_PATH = "/kaggle/input/datasets/basilsaji03/repo-files"

H2S_CKPT = "/kaggle/input/datasets/basilsaji03/unisign-dataset/how2sign_pose_only_slt.pth"

Cell 3 — Load Split

In [ ]:
split = np.load(
    f"{MEETIX_DATASET}/full_train.npz",
    allow_pickle=True
)

print(split.files)

print(len(split["paths"]))
print(len(split["labels"]))

In [ ]:
with open(
    f"{MEETIX_DATASET}/phrase_map.json"
) as f:

    phrase_map = json.load(f)

id_to_phrase = {
    v:k
    for k,v in phrase_map.items()
}

print(id_to_phrase)

In [ ]:
import json

with open(
    f"{MEETIX_DATASET}/vocab_info.json"
) as f:

    vocab_info = json.load(f)

print(vocab_info)

In [ ]:
sample = np.load(
    split["paths"][0]
)["landmarks"]

print(sample.shape)

print(sample[0][:30])

In [ ]:
sample = np.load(
    split["paths"][0]
)["landmarks"]

print(sample.shape)

frame = sample[0]

print(len(frame))

face = frame[:1404]
pose = frame[1404:1536]
left = frame[1536:1599]
right = frame[1599:1662]

print("face :", face.shape)
print("pose :", pose.shape)
print("left :", left.shape)
print("right:", right.shape)

Cell 5 — Add Repo

In [ ]:
import sys

sys.path.append(REPO_PATH)

# Cell 6 — Encoder Definition

In [ ]:
import torch
from torch import nn

# 2. Import directly from the flat files instead of the stgcn_layers folder
from gcn_utils import Graph
from stgcn_block import get_stgcn_chain

class UniSignEncoderOnly(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.modes = ['body', 'left', 'right', 'face_all']
        self.graph, A = {}, []

        self.proj_linear = nn.ModuleDict()
        for mode in self.modes:
            self.graph[mode] = Graph(layout=f'{mode}', strategy='distance', max_hop=1)
            A.append(torch.tensor(self.graph[mode].A, dtype=torch.float32, requires_grad=False))
            self.proj_linear[mode] = nn.Linear(3, 64)

        self.gcn_modules = nn.ModuleDict()
        self.fusion_gcn_modules = nn.ModuleDict()
        spatial_kernel_size = A[0].size(0)

        for index, mode in enumerate(self.modes):
            self.gcn_modules[mode], final_dim = get_stgcn_chain(64, 'spatial', (1, spatial_kernel_size), A[index].clone(), True)
            self.fusion_gcn_modules[mode], _ = get_stgcn_chain(final_dim, 'temporal', (5, spatial_kernel_size), A[index].clone(), True)

        self.gcn_modules['left'] = self.gcn_modules['right']
        self.fusion_gcn_modules['left'] = self.fusion_gcn_modules['right']
        self.proj_linear['left'] = self.proj_linear['right']

        self.part_para = nn.Parameter(torch.zeros(hidden_dim * len(self.modes)))
        self.pose_proj = nn.Linear(256 * 4, 768)

    def forward(self, src_input):
        features = []
        body_feat = None

        for part in self.modes:
            proj_feat = self.proj_linear[part](src_input[part]).permute(0,3,1,2) # B,C,T,V
            gcn_feat = self.gcn_modules[part](proj_feat)

            if part == 'body':
                body_feat = gcn_feat
            else:
                if part == 'left':
                    gcn_feat = gcn_feat + body_feat[..., -2][...,None].detach()
                elif part == 'right':
                    gcn_feat = gcn_feat + body_feat[..., -1][...,None].detach()
                elif part == 'face_all':
                    gcn_feat = gcn_feat + body_feat[..., 0][...,None].detach()

            gcn_feat = self.fusion_gcn_modules[part](gcn_feat)
            pool_feat = gcn_feat.mean(-1).transpose(1,2) # B,T,C
            features.append(pool_feat)

        inputs_embeds = torch.cat(features, dim=-1) + self.part_para
        inputs_embeds = self.pose_proj(inputs_embeds)

        return inputs_embeds # Output: (B, T, 768)

Cell 7 — Load H2S Weights

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

encoder = UniSignEncoderOnly()

sd = torch.load(
    H2S_CKPT,
    map_location=device
)["model"]

msg = encoder.load_state_dict(
    sd,
    strict=False
)

print(msg)

encoder = encoder.to(device)
encoder.eval()

Cell 8 - Uni-Sign Adapter

In [ ]:
import numpy as np
import torch

def mediapipe_to_unisign(sequence):
    """
    sequence:
        (T,1662)

    returns:
        {
            body      (1,T,9,3)
            left      (1,T,21,3)
            right     (1,T,21,3)
            face_all  (1,T,18,3)
        }
    """

    T = sequence.shape[0]

    # --------------------------------------------------
    # Split modalities
    # --------------------------------------------------

    face = sequence[:, :1404].reshape(T, 468, 3)

    pose = sequence[:, 1404:1536].reshape(
        T,
        33,
        4
    )[:, :, :3]

    left = sequence[:, 1536:1599].reshape(
        T,
        21,
        3
    )

    right = sequence[:, 1599:1662].reshape(
        T,
        21,
        3
    )

    # --------------------------------------------------
    # UniSign expects 9 body joints
    # --------------------------------------------------

    body_idx = [
        0,   # nose
        11,  # left shoulder
        12,  # right shoulder
        13,  # left elbow
        14,  # right elbow
        15,  # left wrist
        16,  # right wrist
        23,  # left hip
        24   # right hip
    ]

    body = pose[:, body_idx]

    # --------------------------------------------------
    # Face compression
    # --------------------------------------------------

    face_idx = np.linspace(
        0,
        467,
        18
    ).astype(int)

    face18 = face[:, face_idx]

    # --------------------------------------------------
    # Root normalization
    # --------------------------------------------------

    left = left - left[:, 0:1]
    right = right - right[:, 0:1]

    face18 = face18 - face18[:, -1:]

    # shoulders midpoint
    root = (
        body[:, 1:2] +
        body[:, 2:3]
    ) / 2

    body = body - root

    # --------------------------------------------------
    # Batch dimension
    # --------------------------------------------------

    src = {

        "body":
            torch.tensor(body)
            .float()
            .unsqueeze(0),

        "left":
            torch.tensor(left)
            .float()
            .unsqueeze(0),

        "right":
            torch.tensor(right)
            .float()
            .unsqueeze(0),

        "face_all":
            torch.tensor(face18)
            .float()
            .unsqueeze(0)
    }

    return src

Cell 9 - Test adapter.

In [ ]:
sample = np.load(
    split["paths"][0]
)["landmarks"]

src = mediapipe_to_unisign(
    sample
)


for k in src:
    print(
        k,
        src[k].shape
    )

CELL 10

In [ ]:
for k in src:
    src[k] = src[k].to(device)

with torch.no_grad():

    seq = encoder(src)

print(seq.shape)

In [ ]:
print(next(encoder.parameters()).device)

for k in src:
    print(k, src[k].device)

# Cell 11 — Build Label Mapping

In [ ]:
PHRASE_TO_TOKENS = {

    "CAN I HELP YOU":
        ["CAN","I","HELP_YOU","YOU"],

    "CAN YOU HELP ME":
        ["CAN","YOU","HELP_ME"],

    "CAN YOU REPEAT":
        ["CAN","YOU","REPEAT"],

    "CAN YOU SEE ME":
        ["CAN","YOU","SEE","ME"],

    "CAN YOU WAIT":
        ["CAN","YOU","WAIT"],

    "I NEED HELP":
        ["I","NEED","HELP_ME"],

    "I NOT UNDERSTAND":
        ["I","NOT","UNDERSTAND"],

    "I UNDERSTAND":
        ["I","UNDERSTAND"],

    "PLEASE HELP ME":
        ["PLEASE","HELP_ME"],

    "PLEASE REPEAT":
        ["PLEASE","REPEAT"],

    "PLEASE WAIT":
        ["PLEASE","WAIT"],

    "YOU NEED HELP":
        ["YOU","NEED","HELP_YOU"],

    "YOU UNDERSTAND":
        ["YOU","UNDERSTAND"]
}

In [ ]:
special = [
    "<PAD>",
    "<SOS>",
    "<EOS>"
]

tokens = set()

for seq in PHRASE_TO_TOKENS.values():
    tokens.update(seq)

VOCAB = special + sorted(tokens)

token_to_id = {
    t:i
    for i,t in enumerate(VOCAB)
}

id_to_token = {
    i:t
    for t,i in token_to_id.items()
}

VOCAB_SIZE = len(VOCAB)

print(VOCAB_SIZE)
print(VOCAB)

In [ ]:
print("VOCAB SIZE:", VOCAB_SIZE)

for i,t in enumerate(VOCAB):
    print(i, t)

Cell 12 — Load Training Split - FULL SPLIT

In [ ]:
train_split = np.load(
    f"{MEETIX_DATASET}/full_train.npz",
    allow_pickle=True
)

val_split = np.load(
    f"{MEETIX_DATASET}/full_val.npz",
    allow_pickle=True
)

print(len(train_split["paths"]))
print(len(val_split["paths"]))

Cell 13 — Dataset Class

In [ ]:
class MeetixDataset(Dataset):

    def __init__(self, split):

        self.paths = split["paths"]
        self.labels = split["labels"]
        self.phrases = split["phrases"]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        arr = np.load(self.paths[idx])["landmarks"]

        src = mediapipe_to_unisign(arr)

        # =========================
        # REPLACED LABEL LOGIC
        # =========================

        phrase = str(
            self.phrases[idx]
        )

        tokens = PHRASE_TO_TOKENS[
            phrase
        ]

        token_ids = [
            token_to_id["<SOS>"]
        ]

        token_ids += [
            token_to_id[t]
            for t in tokens
        ]

        token_ids.append(
            token_to_id["<EOS>"]
        )

        return src, torch.tensor(token_ids, dtype=torch.long)

In [ ]:
print(train_split.files)

In [ ]:
print(train_split.files)
print(train_split["phrases"][:5])

Cell 14 — Padding Function

In [ ]:
def collate_fn(batch):

    srcs, labels = zip(*batch)

    max_len = max(
        x["body"].shape[1]
        for x in srcs
    )

    out = {}

    for key in [
        "body",
        "left",
        "right",
        "face_all"
    ]:

        padded = []

        for s in srcs:

            x = s[key]

            T = x.shape[1]

            if T < max_len:

                pad = torch.zeros(
                    (
                        1,
                        max_len - T,
                        x.shape[2],
                        x.shape[3]
                    ),
                    dtype=x.dtype
                )

                x = torch.cat([x, pad], dim=1)

            padded.append(x)

        out[key] = torch.cat(padded, dim=0)

    # =========================
    # TOKEN SEQUENCE PADDING
    # =========================

    max_tok_len = max(
        len(t)
        for t in labels
    )

    tok_batch = []

    for seq in labels:

        if len(seq) < max_tok_len:

            pad = torch.full(
                (max_tok_len - len(seq),),
                token_to_id["<PAD>"]
            )

            seq = torch.cat([seq, pad])

        tok_batch.append(seq)

    tok_batch = torch.stack(tok_batch).long()

    return out, tok_batch

Cell 15 — Create Dataloaders

In [ ]:
train_ds = MeetixDataset(
    train_split
)

val_ds = MeetixDataset(
    val_split
)

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn
)

Cell 16 — Verify Batch

In [ ]:
batch = next(
    iter(train_loader)
)

src, labels = batch

for k in src:
    print(
        k,
        src[k].shape
    )

print(labels.shape)

# Cell 17 — Decoder Head

In [ ]:
class MeetixSeq2Seq(nn.Module):

    def __init__(self, encoder):

        super().__init__()

        self.encoder = encoder

        self.proj = nn.Linear(
            768,
            256
        )

        self.bigru = nn.GRU(
            256,
            128,
            batch_first=True,
            bidirectional=True
        )

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            128
        )

        self.decoder = nn.GRU(
            128 + 256,
            256,
            batch_first=True
        )

        self.out = nn.Linear(
            256,
            VOCAB_SIZE
        )

    def forward(self, src, tgt):

        enc = self.encoder(src)

        enc = self.proj(enc)

        enc, _ = self.bigru(enc)

        context = enc.mean(1)

        emb = self.embedding(
            tgt[:, :-1]
        )

        context = context.unsqueeze(1)

        context = context.repeat(
            1,
            emb.shape[1],
            1
        )

        dec_in = torch.cat(
            [emb, context],
            dim=-1
        )

        out, _ = self.decoder(dec_in)

        logits = self.out(out)

        return logits

Cell 18 — Freeze Encoder

In [ ]:
for p in encoder.parameters():
    p.requires_grad = False

encoder.eval()

# Cell 19 — Create Model

In [ ]:
model = MeetixSeq2Seq(
    encoder
).to(device)

print(
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

# Cell 20 — Single Forward Pass

In [ ]:
src, labels = next(
    iter(train_loader)
)

for k in src:
    src[k] = src[k].to(device)

labels = labels.to(device)

with torch.no_grad():

    logits = model(src, labels)

print(logits.shape)

In [ ]:
print("Labels shape:", labels.shape)
print("Logits shape:", logits.shape)
print("Vocab size:", VOCAB_SIZE)

Cell 21 — Loss

In [ ]:
criterion = nn.CrossEntropyLoss(
    ignore_index=
    token_to_id["<PAD>"]
)

Cell 22 — Optimizer

In [ ]:
optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

Cell 23 — Accuracy Function

In [ ]:
def token_accuracy(
    logits,
    labels
):

    preds = logits.argmax(-1)

    target = labels[:,1:]

    mask = (
        target != token_to_id["<PAD>"]
    )

    correct = (
        preds == target
    ) & mask

    return (
        correct.sum().float()
        /
        mask.sum().float()
    ).item()

Cell 24 — Validation Function

In [ ]:
def evaluate(model, loader):

    model.eval()

    losses = []
    accs = []

    with torch.no_grad():

        for src, labels in loader:

            for k in src:
                src[k] = src[k].to(device)

            labels = labels.to(device)

            logits = model(
                src,
                labels
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                labels[:, 1:].reshape(-1)
            )

            losses.append(
                loss.item()
            )

            accs.append(
                token_accuracy(
                    logits,
                    labels
                )
            )

    return (
        np.mean(losses),
        np.mean(accs)
    )

# Cell 25 — Training Loop

In [ ]:
best_val_acc = 0

history = []

EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    train_losses = []
    train_accs = []

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for src, labels in pbar:

        for k in src:
            src[k] = src[k].to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(
            src,
            labels
        )

        loss = criterion(
            logits.reshape(
                -1,
                VOCAB_SIZE
            ),
            labels[:,1:].reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        acc = token_accuracy(
            logits,
            labels
        )

        train_losses.append(
            loss.item()
        )

        train_accs.append(
            acc
        )

        pbar.set_postfix(
            loss=np.mean(train_losses),
            acc=np.mean(train_accs)
        )

    val_loss, val_acc = evaluate(
        model,
        val_loader
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": np.mean(train_losses),
        "train_acc": np.mean(train_accs),
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    print(
        f"\nEpoch {epoch+1}"
        f" | Train Acc {np.mean(train_accs):.4f}"
        f" | Val Acc {val_acc:.4f}"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_meetix_seq2seq.pth"
        )

        print(
            f"Saved new best model "
            f"(Val Acc={val_acc:.4f})"
        )

In [ ]:
def decode_tokens(token_ids):

    out = []

    for t in token_ids:

        tok = id_to_token[int(t)]

        if tok in ["<PAD>", "<SOS>"]:
            continue

        if tok == "<EOS>":
            break

        out.append(tok)

    return out

In [ ]:
model.eval()

src, labels = next(iter(val_loader))

for k in src:
    src[k] = src[k].to(device)

labels = labels.to(device)

with torch.no_grad():

    logits = model(src, labels)

preds = logits.argmax(-1)

for i in range(min(20, len(preds))):

    gt = decode_tokens(
        labels[i].cpu().numpy()
    )

    pred = decode_tokens(
        preds[i].cpu().numpy()
    )

    print("GT  :", gt)
    print("PRED:", pred)
    print("-"*60)

In [ ]:
import os
import json
import torch

SAVE_DIR = "/kaggle/working/meetix_seq2seq"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

torch.save(
    model.state_dict(),
    f"{SAVE_DIR}/best_meetix_seq2seq.pth"
)

with open(
    f"{SAVE_DIR}/vocab.json",
    "w"
) as f:

    json.dump(
        {
            "VOCAB": VOCAB,
            "token_to_id": token_to_id
        },
        f,
        indent=2
    )

print("Saved.")

In [ ]:
def decode_tokens(token_ids):

    out = []

    for t in token_ids:

        tok = id_to_token[int(t)]

        if tok in ["<PAD>", "<SOS>"]:
            continue

        if tok == "<EOS>":
            break

        out.append(tok)

    return out

In [ ]:
model.eval()

correct = 0
total = 0

src, labels = next(iter(val_loader))

for k in src:
    src[k] = src[k].to(device)

labels = labels.to(device)

with torch.no_grad():

    logits = model(
        src,
        labels
    )

preds = logits.argmax(-1)

for i in range(len(preds)):

    gt = decode_tokens(
        labels[i].cpu().numpy()
    )

    pred = decode_tokens(
        preds[i].cpu().numpy()
    )

    if gt == pred:
        correct += 1

    total += 1

print(
    "Phrase Accuracy:",
    correct / total
)

In [ ]:
model.eval()

correct = 0
total = 0

for src, labels in val_loader:

    for k in src:
        src[k] = src[k].to(device)

    labels = labels.to(device)

    with torch.no_grad():

        logits = model(
            src,
            labels
        )

    preds = logits.argmax(-1)

    for i in range(len(preds)):

        gt = decode_tokens(
            labels[i].cpu().numpy()
        )

        pred = decode_tokens(
            preds[i].cpu().numpy()
        )

        if gt == pred:
            correct += 1

        total += 1

print(
    f"Phrase Accuracy: {correct/total:.4f}"
)

In [ ]:
rows = []

model.eval()

for src, labels in val_loader:

    for k in src:
        src[k] = src[k].to(device)

    labels = labels.to(device)

    with torch.no_grad():

        logits = model(
            src,
            labels
        )

    preds = logits.argmax(-1)

    for i in range(len(preds)):

        gt = " ".join(
            decode_tokens(
                labels[i].cpu().numpy()
            )
        )

        pred = " ".join(
            decode_tokens(
                preds[i].cpu().numpy()
            )
        )

        rows.append(
            {
                "gt": gt,
                "pred": pred,
                "correct": gt == pred
            }
        )

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)

df.to_csv(
    f"{SAVE_DIR}/prediction_report.csv",
    index=False
)

df.head()

In [ ]:
wrong = df[
    df["correct"] == False
]

print(len(wrong))

wrong.head(50)

In [ ]:
import os

SAVE_DIR = "/kaggle/working/meetix_seq2seq_full"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

print(SAVE_DIR)

In [ ]:
torch.save(
    model.state_dict(),
    f"{SAVE_DIR}/best_meetix_seq2seq.pth"
)

In [ ]:
torch.save(
    encoder.state_dict(),
    f"{SAVE_DIR}/h2s_encoder.pth"
)

In [ ]:
import json

with open(
    f"{SAVE_DIR}/vocab.json",
    "w"
) as f:

    json.dump(
        {
            "VOCAB": VOCAB,
            "token_to_id": token_to_id,
            "id_to_token": id_to_token
        },
        f,
        indent=2
    )

In [ ]:
df.to_csv(
    f"{SAVE_DIR}/prediction_report.csv",
    index=False
)

In [ ]:
wrong = df[
    df["correct"] == False
]

wrong.to_csv(
    f"{SAVE_DIR}/wrong_predictions.csv",
    index=False
)

print(len(wrong))

In [ ]:
summary = f"""
Meetix Seq2Seq Decoder Results

Dataset:
FULL SPLIT

Vocabulary Size:
{VOCAB_SIZE}

Phrase Accuracy:
0.9048

Failure Count:
{len(wrong)}

Notable Failure Modes:
I NOT UNDERSTAND -> I UNDERSTAND UNDERSTAND
YOU NEED HELP_YOU -> I NEED HELP_YOU
"""

with open(
    f"{SAVE_DIR}/results_summary.txt",
    "w"
) as f:

    f.write(summary)

In [ ]:
with open(
    f"{SAVE_DIR}/phrase_to_tokens.json",
    "w"
) as f:

    json.dump(
        PHRASE_TO_TOKENS,
        f,
        indent=2
    )

In [ ]:
config = {
    "vocab_size": VOCAB_SIZE,
    "hidden_size": 256,
    "gru_hidden": 128,
    "encoder": "UniSign H2S",
    "decoder": "BiGRU + GRU Seq2Seq",
    "split": "FULL",
    "phrase_accuracy": 0.9048
}

with open(
    f"{SAVE_DIR}/config.json",
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )

In [ ]:
import os

for f in sorted(os.listdir(SAVE_DIR)):
    print(f)